# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christianabusca/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone https://github.com/christianabusca/flyrank-ml-internship.git
%cd flyrank-ml-internship
!ls
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 156 (delta 63), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 1.87 MiB | 11.76 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/flyrank-ml-internship/flyrank-ml-internship
AGENTS.md    docs	outputs		  SETUP.md		     work
CLAUDE.md    GUIDE.md	README.md	  skills
data	     LICENSE	requirements.txt  submission
DATA_USE.md  notebooks	scripts		  w02_ml_task_framing.ipynb


In [9]:
# Step 1: Load the starter data and apply the starter filters
print("raw rows:", len(df))

# Same filters the starter pipeline uses
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates("content_id")
print("after filters:", len(df))

raw rows: 30000
after filters: 30000


In [8]:
# Step 2: Unit of analysis as a real dataframe (Section 4 of the notebook)
lane_cols = [
    "content_id", "client_id",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "impressions_last_30d", "impressions_prev_30d",
    "trend_pct", "trend_direction",
]
lane_df = df[lane_cols].copy()
lane_df.head(10)

# Grain check: should print 0 (no repeated content_id)
print(lane_df["content_id"].duplicated().sum())

0


In [10]:
# Step 3: Look at the built-in label (Section 2: target or proxy)
print(lane_df["trend_direction"].value_counts())
print()
print(lane_df.groupby("trend_direction")["trend_pct"].describe())
# Missing values in trend_pct
print("trend_pct missing:", lane_df["trend_pct"].isna().sum())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

                   count        mean          std    min   25%    50%    75%  \
trend_direction                                                                
down             16262.0  -58.113830    23.488605 -100.0 -75.9 -55.60  -38.5   
flat                 0.0         NaN          NaN    NaN   NaN    NaN    NaN   
new                  0.0         NaN          NaN    NaN   NaN    NaN    NaN   
stable            5962.0   -3.185944    11.054723  -20.0 -12.7  -3.80    5.0   
up                4388.0  190.673997  1145.029477   20.0  36.5  62.55  123.1   

                     max  
trend_direction           
down               -20.0  
flat                 NaN  
new                  NaN  
stable              20.0  
up               44900.0  
trend_pct missing: 3388


In [11]:
# Step 4: Sketch your own proxy from the 30-day columns
# Only pages that had some traffic in the previous 30 days (avoids dividing by zero)
base = lane_df[lane_df["impressions_prev_30d"] > 0].copy()

# Percent change in impressions, last 30 days vs the 30 days before
base["impr_change_pct"] = (
    (base["impressions_last_30d"] - base["impressions_prev_30d"])
    / base["impressions_prev_30d"] * 100
)

print(len(base), "rows with prev-30d impressions > 0")
base["impr_change_pct"].describe()

# Compare with the built-in trend_pct. Do they agree?
base[["impr_change_pct", "trend_pct"]].corr()

# Draft target column. The threshold below is a PLACEHOLDER. Decide your own and write down why.
THRESHOLD = -20  # percent
base["my_proxy_declining"] = (base["impr_change_pct"] <= THRESHOLD).astype(int)

print(base["my_proxy_declining"].value_counts())
print("share declining:", base["my_proxy_declining"].mean().round(3))
base[["content_id", "impressions_prev_30d", "impressions_last_30d",
      "impr_change_pct", "my_proxy_declining"]].head(10)

26612 rows with prev-30d impressions > 0
my_proxy_declining
1    16305
0    10307
Name: count, dtype: int64
share declining: 0.613


,content_id,impressions_prev_30d,impressions_last_30d,impr_change_pct,my_proxy_declining
0,content_304f48230142,987,578,-41.438703,1
1,content_a1fb4e703a9e,5915,2501,-57.717667,1
2,content_9aa793d4d895,6089,2382,-60.880276,1
3,content_331d6c4de07b,4206,3626,-13.789824,0
4,content_d99b7a2d90ca,6452,4211,-34.733416,1
5,content_d4084a4bc775,1009,617,-38.850347,1
6,content_9a34b442b552,13,1,-92.307692,1
7,content_a63219c6e95a,632,636,0.632911,0
8,content_5e6c160719bc,13828,5696,-58.808215,1
9,content_c27558df2b0c,356,252,-29.213483,1


In [12]:
[c for c in df.columns if "30d" in c or "next" in c or "future" in c]

['impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d']

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is a ranking/scoring task, because my output is an ordered list of pages to review, not a yes/no answer.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target (proxy) is trend_direction == "down"

It is only a proxy because it looks at what already happened in the past, rather than measuring a true future outcome.

A stronger target would be looking at declines over the next 30 days, which needs the daily data from the full warehouse.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is Precision@50, because a reviewer can only check a limited number of pages, so we need to measure the accuracy of the model's top recommendations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

“One row = one page, identified by content_id. The check printed 0 duplicates, so every page appears once.”

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like 'old AND traffic dropped a lot' would wrongly flag Page A for a normal traffic dip, while completely missing Page B's alarming drop from position #3 to #15. A machine learning model is better because it evaluates how multiple signals interact—weighing changes in avg_position alongside impressions_90d and ctr—to accurately recognize that Page B is the one actually losing search visibility and needs an urgent review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.